# Phase 4: Predictive Modeling

In this notebook, we train, evaluate, and serialize machine learning models for the 3 pillars:
1. **Model 1: Microplastic Risk Predictor** (Regression - Ridge vs. XGBoost Regressor on log-transformed particle counts).
2. **Model 2: Waste Cleanup Yield Predictor** (Regression - Ridge vs. XGBoost Regressor on log-transformed cleanup tonnage).
3. **Model 3: Policy Success Classifier** (Classification - Logistic Regression vs. XGBoost Classifier on binary success target).

We evaluate performance using MSE, RMSE, R2 (regression) and Accuracy, Recall, F1, ROC-AUC (classification), and serialize the best estimators.

## 1. Setup Environment & Load Processed Data

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBRegressor, XGBClassifier

processed_dir = "../data/processed/"
models_dir = "../models/"
os.makedirs(models_dir, exist_ok=True)

# Load Pillar 1 (Microplastics)
X_train_micro = pd.read_csv(os.path.join(processed_dir, "X_train_micro.csv"))
X_test_micro = pd.read_csv(os.path.join(processed_dir, "X_test_micro.csv"))
y_train_micro = pd.read_csv(os.path.join(processed_dir, "y_train_micro.csv")).values.ravel()
y_test_micro = pd.read_csv(os.path.join(processed_dir, "y_test_micro.csv")).values.ravel()

# Load Pillar 2 (Waste Cleanup)
X_train_waste = pd.read_csv(os.path.join(processed_dir, "X_train_waste.csv"))
X_test_waste = pd.read_csv(os.path.join(processed_dir, "X_test_waste.csv"))
y_train_waste = pd.read_csv(os.path.join(processed_dir, "y_train_waste.csv")).values.ravel()
y_test_waste = pd.read_csv(os.path.join(processed_dir, "y_test_waste.csv")).values.ravel()

# Load Pillar 3 (Green Initiatives)
X_train_green = pd.read_csv(os.path.join(processed_dir, "X_train_green.csv"))
X_test_green = pd.read_csv(os.path.join(processed_dir, "X_test_green.csv"))
y_train_green = pd.read_csv(os.path.join(processed_dir, "y_train_green.csv")).values.ravel()
y_test_green = pd.read_csv(os.path.join(processed_dir, "y_test_green.csv")).values.ravel()

print("All train and test sets loaded successfully.")

All train and test sets loaded successfully.


## 2. Model 1: Microplastic Risk Predictor (Regression)

We train baseline Ridge Regression and XGBoost Regressor to predict `log(concentration + 1)`.

In [2]:
# Train Baseline Ridge
lr_micro = Ridge(alpha=1.0)
lr_micro.fit(X_train_micro, y_train_micro)
y_pred_lr = lr_micro.predict(X_test_micro)

# Train XGBoost
xgb_micro = XGBRegressor(n_estimators=100, learning_rate=0.08, max_depth=6, random_state=42, n_jobs=-1)
xgb_micro.fit(X_train_micro, y_train_micro)
y_pred_xgb = xgb_micro.predict(X_test_micro)

# Evaluate (Log space)
print("=== Ridge Regression (Microplastics) ===")
print(f"MAE: {mean_absolute_error(y_test_micro, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_micro, y_pred_lr)):.4f}")
print(f"R2 Score: {r2_score(y_test_micro, y_pred_lr):.4f}")

print("\n=== XGBoost Regressor (Microplastics) ===")
print(f"MAE: {mean_absolute_error(y_test_micro, y_pred_xgb):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_micro, y_pred_xgb)):.4f}")
print(f"R2 Score: {r2_score(y_test_micro, y_pred_xgb):.4f}")

# Serialize the best model
with open(os.path.join(models_dir, "model_micro.pkl"), "wb") as f:
    pickle.dump(xgb_micro, f)
print("\nMicroplastics model saved to models/model_micro.pkl")

=== Ridge Regression (Microplastics) ===
MAE: 0.8554
RMSE: 1.0772
R2 Score: 0.2273

=== XGBoost Regressor (Microplastics) ===
MAE: 0.3743
RMSE: 0.4706
R2 Score: 0.8525

Microplastics model saved to models/model_micro.pkl


## 3. Model 2: Waste Cleanup Yield Predictor (Regression)

We train baseline Ridge and XGBoost to predict `log(weight_collected_ton + 1)`.

In [3]:
# Train Baseline Ridge
lr_waste = Ridge(alpha=1.0)
lr_waste.fit(X_train_waste, y_train_waste)
y_pred_lr = lr_waste.predict(X_test_waste)

# Train XGBoost
xgb_waste = XGBRegressor(n_estimators=100, learning_rate=0.08, max_depth=6, random_state=42, n_jobs=-1)
xgb_waste.fit(X_train_waste, y_train_waste)
y_pred_xgb = xgb_waste.predict(X_test_waste)

# Evaluate
print("=== Ridge Regression (Waste Yield) ===")
print(f"MAE: {mean_absolute_error(y_test_waste, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_waste, y_pred_lr)):.4f}")
print(f"R2 Score: {r2_score(y_test_waste, y_pred_lr):.4f}")

print("\n=== XGBoost Regressor (Waste Yield) ===")
print(f"MAE: {mean_absolute_error(y_test_waste, y_pred_xgb):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_waste, y_pred_xgb)):.4f}")
print(f"R2 Score: {r2_score(y_test_waste, y_pred_xgb):.4f}")

# Serialize model
with open(os.path.join(models_dir, "model_waste.pkl"), "wb") as f:
    pickle.dump(xgb_waste, f)
print("\nWaste model saved to models/model_waste.pkl")

=== Ridge Regression (Waste Yield) ===
MAE: 0.8346
RMSE: 1.0099
R2 Score: 0.0895

=== XGBoost Regressor (Waste Yield) ===
MAE: 0.1289
RMSE: 0.1913
R2 Score: 0.9673

Waste model saved to models/model_waste.pkl


## 4. Model 3: Policy Success Classifier (Binary Classification)

We train Logistic Regression and XGBoost Classifier to classify if `achieved_policy_impact` is 1 or 0.

In [4]:
# Train Logistic Regression
clf_lr = LogisticRegression(max_iter=500, random_state=42)
clf_lr.fit(X_train_green, y_train_green)
y_pred_lr = clf_lr.predict(X_test_green)
y_prob_lr = clf_lr.predict_proba(X_test_green)[:, 1]

# Train XGBoost Classifier
xgb_green = XGBClassifier(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=42, n_jobs=-1)
xgb_green.fit(X_train_green, y_train_green)
y_pred_xgb = xgb_green.predict(X_test_green)
y_prob_xgb = xgb_green.predict_proba(X_test_green)[:, 1]

# Evaluate
print("=== Logistic Regression (Policy Success) ===")
print(f"Accuracy: {accuracy_score(y_test_green, y_pred_lr):.4f}")
print(f"F1 Score: {f1_score(y_test_green, y_pred_lr):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_green, y_prob_lr):.4f}")

print("\n=== XGBoost Classifier (Policy Success) ===")
print(f"Accuracy: {accuracy_score(y_test_green, y_pred_xgb):.4f}")
print(f"F1 Score: {f1_score(y_test_green, y_pred_xgb):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_green, y_prob_xgb):.4f}")

# Serialize model
with open(os.path.join(models_dir, "model_green.pkl"), "wb") as f:
    pickle.dump(xgb_green, f)
print("\nPolicy model saved to models/model_green.pkl")

=== Logistic Regression (Policy Success) ===
Accuracy: 0.5271
F1 Score: 0.2651
ROC-AUC: 0.4859

=== XGBoost Classifier (Policy Success) ===
Accuracy: 0.5141
F1 Score: 0.3381
ROC-AUC: 0.5094

Policy model saved to models/model_green.pkl
